# Ejecutor de dbt — Capa Silver y Gold

Este notebook ejecuta el proyecto **dbt** que vive en la carpeta `dbt/` del repo,
directamente desde Databricks (sin necesidad de instalar dbt en tu PC local).

## Qué hace este notebook

1. Instala `dbt-core` y `dbt-databricks` en el cluster.
2. Genera dinámicamente el archivo `profiles.yml` con las credenciales del cluster actual.
3. Ejecuta `dbt build` (corre Silver → Gold + tests).
4. Muestra el resumen de la ejecución.

## Cómo se conecta dbt con Databricks desde aquí

A diferencia de ejecutar dbt en tu PC (que requiere host + http_path + token),
cuando dbt corre **dentro** de Databricks usa las credenciales del cluster
automáticamente vía el token implícito del notebook.

## Para la demo en vivo

Simplemente: **Run all** en este notebook. dbt construye Silver y Gold en vivo,
se ven los logs en tiempo real, y al final las tablas `silver.silver_*` y
`gold.gold_*` quedan reconstruidas por dbt.

## 1. Configuración — variables del proyecto

In [ ]:
import os

# ============================================================
# 🔒 CONFIGURACIÓN VÍA WIDGETS (sin credenciales en Git)
# ============================================================
# Crea los widgets si no existen. Los valores que pongas
# quedan guardados en TU Databricks, NO en el repositorio Git.
#
# La PRIMERA VEZ tienes que llenar las 3 cajas arriba del notebook:
#   1. databricks_host   — ej. dbc-abc123.cloud.databricks.com
#   2. warehouse_http_path — ej. /sql/1.0/warehouses/xxxxx
#   3. databricks_token  — tu PAT (dapi...)
#
# Después de llenarlos, dale Run all. Los widgets persisten
# entre ejecuciones — no tienes que volver a llenarlos.
# ============================================================

dbutils.widgets.text("databricks_host", "", "1. Databricks Host (sin https://)")
dbutils.widgets.text("warehouse_http_path", "", "2. SQL Warehouse HTTP Path")
dbutils.widgets.text("databricks_token", "", "3. Personal Access Token (dapi...)")

# Leer valores de los widgets
DATABRICKS_HOST = dbutils.widgets.get("databricks_host")
WAREHOUSE_HTTP_PATH = dbutils.widgets.get("warehouse_http_path")
DATABRICKS_TOKEN = dbutils.widgets.get("databricks_token")

# Validar
if not DATABRICKS_HOST or not WAREHOUSE_HTTP_PATH or not DATABRICKS_TOKEN:
    raise ValueError(
        "⚠️ Llena las 3 cajas arriba del notebook con tus credenciales. "
        "Aparecen como widgets bajo el título del notebook. "
        "Una vez llenas, dale Run all otra vez."
    )

# ============================================================
# Variables del proyecto (estas SÍ pueden estar en Git)
# ============================================================
CATALOG_NAME = spark.sql("SELECT current_catalog()").collect()[0][0]
DBT_PROJECT_DIR = "/Workspace/Users/orojanom@unicesar.edu.co/wanderbricks-lakehouse-project/dbt"
DBT_PROFILES_DIR = "/tmp/dbt_profiles"

# Exportar variables Python al shell para que las celdas %sh las vean
os.environ["DBT_PROJECT_DIR"] = DBT_PROJECT_DIR
os.environ["DBT_PROFILES_DIR"] = DBT_PROFILES_DIR
os.environ["CATALOG_NAME"] = CATALOG_NAME

print(f"Catálogo detectado: {CATALOG_NAME}")
print(f"Proyecto dbt:       {DBT_PROJECT_DIR}")
print(f"Profiles dir:       {DBT_PROFILES_DIR}")
print(f"Host:               {DATABRICKS_HOST}")
print(f"SQL Warehouse:      {WAREHOUSE_HTTP_PATH}")
print(f"Token:              ***{DATABRICKS_TOKEN[-4:]} (últimos 4 chars)")
print()
print("✓ Configuración leída desde widgets (NO desde Git)")

## 2. Instalar dbt-core + dbt-databricks en el cluster

Esto solo tarda 30-60 segundos la primera vez. Después de instalado, las próximas ejecuciones lo saltean.

In [ ]:
%pip install dbt-core dbt-databricks --quiet

In [ ]:
dbutils.library.restartPython()

## 3. Verificar que dbt está instalado

In [ ]:
%sh dbt --version

## 4. Generar `profiles.yml` dinámicamente

El `profiles.yml` se crea automáticamente con las credenciales del cluster.

**Reemplaza `<TU_WAREHOUSE_ID>` en la celda 1** con el ID de tu SQL Warehouse:

1. En Databricks → SQL Warehouses → tu warehouse activo.
2. Connection details → copia el HTTP Path (algo como `/sql/1.0/warehouses/abc123def456`).
3. Pégalo en la variable `WAREHOUSE_HTTP_PATH` de la celda 1.

In [ ]:
import os

# ============================================================
# Re-leer widgets (necesario porque restartPython() reinició el kernel)
# Las cajas siguen llenas con lo que pusiste antes — no las tocas.
# ============================================================

DATABRICKS_HOST = dbutils.widgets.get("databricks_host")
WAREHOUSE_HTTP_PATH = dbutils.widgets.get("warehouse_http_path")
DATABRICKS_TOKEN = dbutils.widgets.get("databricks_token")

if not DATABRICKS_HOST or not WAREHOUSE_HTTP_PATH or not DATABRICKS_TOKEN:
    raise ValueError(
        "⚠️ Los widgets están vacíos. Lléalos arriba del notebook y reintenta."
    )

# Variables del proyecto (no sensibles)
CATALOG_NAME = spark.sql("SELECT current_catalog()").collect()[0][0]
DBT_PROJECT_DIR = "/Workspace/Users/orojanom@unicesar.edu.co/wanderbricks-lakehouse-project/dbt"
DBT_PROFILES_DIR = "/tmp/dbt_profiles"

# Exportar al shell
os.environ["DBT_PROJECT_DIR"] = DBT_PROJECT_DIR
os.environ["DBT_PROFILES_DIR"] = DBT_PROFILES_DIR
os.environ["CATALOG_NAME"] = CATALOG_NAME

# Crear directorio de profiles
os.makedirs(DBT_PROFILES_DIR, exist_ok=True)

# Generar el profiles.yml dinámicamente con valores de los widgets
profiles_content = f"""
wanderbricks:
  target: dev
  outputs:
    dev:
      type: databricks
      catalog: {CATALOG_NAME}
      schema: silver
      host: {DATABRICKS_HOST}
      http_path: {WAREHOUSE_HTTP_PATH}
      token: {DATABRICKS_TOKEN}
      threads: 4
"""

profiles_path = f"{DBT_PROFILES_DIR}/profiles.yml"
with open(profiles_path, 'w') as f:
    f.write(profiles_content)

print(f"profiles.yml creado en: {profiles_path}")
print("Configuración:")
print(f"  catalog:   {CATALOG_NAME}")
print(f"  host:      {DATABRICKS_HOST}")
print(f"  http_path: {WAREHOUSE_HTTP_PATH}")
print(f"  token:     ***{DATABRICKS_TOKEN[-4:]}")
print()
print("✓ profiles.yml generado con valores desde los widgets")

## 5. Verificar conexión — `dbt debug`

Esto prueba que dbt puede hablar con tu Databricks.

In [ ]:
%sh
cd $DBT_PROJECT_DIR && dbt debug --profiles-dir $DBT_PROFILES_DIR

## 6. Ejecutar la capa Silver con dbt

Construye las 6 tablas `silver.silver_*` desde Bronze.

In [ ]:
%sh
cd $DBT_PROJECT_DIR && dbt run --select silver --profiles-dir $DBT_PROFILES_DIR

## 7. Ejecutar la capa Gold con dbt

Construye las 5 tablas Gold (1 fact + 4 dims) desde Silver.

In [ ]:
%sh
cd $DBT_PROJECT_DIR && dbt run --select gold --profiles-dir $DBT_PROFILES_DIR

## 8. Ejecutar tests de calidad de datos

Los tests `not_null`, `unique`, `accepted_values` y `relationships` definidos en `schema.yml`.

In [ ]:
%sh
cd $DBT_PROJECT_DIR && dbt test --profiles-dir $DBT_PROFILES_DIR

## 9. Generar documentación con linaje

Después de esto, el linaje queda visible para `dbt docs serve` (si se ejecuta localmente).

In [ ]:
%sh
cd $DBT_PROJECT_DIR && dbt docs generate --profiles-dir $DBT_PROFILES_DIR

## 10. Validación — verificar tablas creadas por dbt

In [ ]:
%sql
SHOW TABLES IN silver;

In [ ]:
%sql
SHOW TABLES IN gold;

In [ ]:
%sql
-- Conteo de todas las tablas Silver y Gold creadas por dbt
SELECT 'silver_users'        AS tabla, COUNT(*) AS registros FROM silver.silver_users
UNION ALL
SELECT 'silver_destinations' AS tabla, COUNT(*) AS registros FROM silver.silver_destinations
UNION ALL
SELECT 'silver_properties'   AS tabla, COUNT(*) AS registros FROM silver.silver_properties
UNION ALL
SELECT 'silver_bookings'     AS tabla, COUNT(*) AS registros FROM silver.silver_bookings
UNION ALL
SELECT 'silver_payments'     AS tabla, COUNT(*) AS registros FROM silver.silver_payments
UNION ALL
SELECT 'silver_reviews'      AS tabla, COUNT(*) AS registros FROM silver.silver_reviews
UNION ALL
SELECT 'gold_fact_reservas'  AS tabla, COUNT(*) AS registros FROM gold.gold_fact_reservas
UNION ALL
SELECT 'gold_dim_users'      AS tabla, COUNT(*) AS registros FROM gold.gold_dim_users
UNION ALL
SELECT 'gold_dim_properties' AS tabla, COUNT(*) AS registros FROM gold.gold_dim_properties
UNION ALL
SELECT 'gold_dim_destinations' AS tabla, COUNT(*) AS registros FROM gold.gold_dim_destinations
UNION ALL
SELECT 'gold_dim_time'       AS tabla, COUNT(*) AS registros FROM gold.gold_dim_time
ORDER BY tabla;

## Conclusión

Ejecutaste el flujo completo:

1. ✅ dbt instalado en el cluster
2. ✅ Conexión a Databricks verificada
3. ✅ Silver construido por dbt (6 tablas)
4. ✅ Gold construido por dbt (5 tablas)
5. ✅ Tests de calidad ejecutados
6. ✅ Documentación generada

### Para la sustentación

Este notebook reemplaza la ejecución de dbt desde la PC. Demostrar dbt en vivo
se vuelve trivial: abres este notebook → **Run all** → muestras los logs de
`dbt run` y `dbt test` ejecutándose, y al final el conteo de tablas creadas.

### Decisiones de diseño defendibles

- **¿Por qué ejecutar dbt dentro de Databricks?** Las empresas usan Databricks Jobs/Workflows con dbt task para orquestar pipelines productivos. Ejecutar dbt desde un notebook simula ese patrón sin la complejidad del setup de Workflows.
- **¿Por qué `profiles.yml` se genera dinámicamente?** Para evitar hardcodear credenciales en el repo. El token se obtiene del contexto del notebook, que tiene acceso automático al workspace.
- **¿Por qué `%sh` y no Python?** dbt es una CLI — la forma idiomática de invocarlo es vía shell. Databricks soporta `%sh` para ejecutar comandos en el driver del cluster.